# Scopus 2026 — two-stage Level 1 / Level 2 ticket classification

Classifies every 2026 Scopus ticket into one of the **16 revised Level 1 categories**, then
assigns a **Level 2** value using one dedicated module per Level 1 category.

**Inputs**

| file | role |
|---|---|
| `definitions/Scopus Ticket Rules and Root Cause Descriptions v3.xlsx` — `Classification Rules` | Level 1 definitions and the decision rules (verbatim into the stage-1 prompt) |
| same workbook — `Root Cause Descriptions` | Level 2 values with summaries and SME scenarios |
| `data/Scopus/scopus verbatim cleaned 2026.xlsx` | 11,940 tickets closed Jan–Sep 2026 |

**Revised taxonomy (agreed in the request, applied in section 1 as authored overrides)**

* Level 1 is the revised list of 16 categories. `Account Management` no longer exists: account
  changes (change email, duplicate/merge accounts, delete account, registration guidance,
  registered-email lookup, password questions) are **Access** even when the customer can sign
  in, and requests about personal data go to the new **Data / Privacy Request**.
* Access Level 2 is the revised list of 11 values, plus `Unknown rootcause` (reported
  separately) for Access tickets where no cause is stated anywhere.
* **Scopus AI Access** uses the same 11 + 1 Level 2 values; Access and Scopus AI Access share
  one stage-2 module.
* **Data / Privacy Request** has no Level 2 list — its Level 2 is set by rule, no LLM call.
* Every other Level 1 uses its root causes from the `Root Cause Descriptions` sheet.

## Why two stages

Each Level 1 has its own short Level 2 list. Stage 1 assigns Level 1; tickets are split by
predicted Level 1 **code** and each subset goes to a module whose taxonomy holds only that
category's Level 2 values, so an invalid L1/L2 pair is impossible by construction. Same design
as `07_sd_2026_two_stage.ipynb`.

## Order of operations

| section | what | cost |
|---|---|---|
| 0–3 | setup, taxonomy, definition workbook, build | $0 |
| 4 | **Gate 1** — review the exact prompts | $0 |
| 5–6 | preprocess tickets, profiles, checkpoint guard | $0 |
| 7 | **Gate 3** — 30-ticket smoke test | ~$0.30 |
| 8 | **Gate 4** — 300-ticket sample for SME review + Oracle crosstab | ~$3 |
| 9–11 | full run: stage 1, split + stage 2, merge | ~$120–180, ~4–5 h |
| 12 | deliverable | $0 |

There is **no human-labelled gold set** for Scopus, so the gates check validity (no blank or
invalid codes, every L2 allowed under its L1) and produce a sample for SME review; they do not
measure accuracy. `STOP_BEFORE_FULL_RUN` in section 8 blocks the expensive run until that
sample has been looked at.

**Checkpoint resume ignores prompt changes.** `rca run` resumes from
`runs/checkpoints/<stem>_checkpoint.jsonl`. Section 6 fingerprints every module's prompt and
discards a checkpoint that was produced by a different prompt, so editing a definition and
re-running never hands back stale answers.

## 0. Configuration — edit this cell

In [ ]:
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import threading
import time
import unicodedata
from pathlib import Path

import pandas as pd

# -- Project root discovery (walk up until rca_pipeline/ is found) ----------
_root = Path().absolute()
while not (_root / "definitions").is_dir() and _root.parent != _root:
    _root = _root.parent
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
BASE = _root

assert pd.__version__.startswith('3.'), (
    f'pandas {pd.__version__} — expected 3.x from the project venv. Wrong kernel selected?')

# -- Inputs ----------------------------------------------------------------
TICKETS_XLSX = BASE / 'data' / 'Scopus' / 'scopus verbatim cleaned 2026.xlsx'
RULES_XLSX   = BASE / 'docs' / 'Scopus Ticket Rules and Root Cause Descriptions v3.xlsx'
SHEET_RULES  = 'Classification Rules'
SHEET_RC     = 'Root Cause Descriptions'

# -- Build artefacts -------------------------------------------------------
DEFN_XLSX     = BASE / 'definitions' / 'scopus_2026_pipeline_definition.xlsx'
TAXONOMY_DIR  = BASE / 'generated' / 'taxonomies'
MODULES_DIR   = BASE / 'generated' / 'modules'
CONFIG_BACKUP = BASE / 'generated' / '_backup_pre_scopus2026'

# -- Columns ---------------------------------------------------------------
ID_COL       = 'INCIDENT_REF'
TEXT_COL     = 'Conversation'
SUBJECT_COL  = 'incident_subject'
DATE_COL     = 'incident_closed_datetime'
ORACLE_L1    = 'contact_reason_lvl1_desc'
ORACLE_L2    = 'contact_reason_lvl2_desc'
TARGET_YEAR  = 2026

# -- Level 1: the revised list, in the order agreed (fixes the codes) ------
L1_ORDER = [
    'Access',
    'Scopus AI Access',
    'Performance & Functionality Issues',
    'Author Profile Correction',
    'Citation / Metrics',
    'Unclear / No Query',
    'Funding / Awarded Grants Data',
    'Content / Indexing',
    'Help Article Feedback',
    'Search / Feature Question',
    'Institution Admin / Support',
    'Data / Privacy Request',
    'Subscription / Sales',
    'Non-Scopus / Wrong Product',
    'ORCID Issue',
    'API / Integration',
]

# -- Access Level 2: the revised list, in the order agreed -----------------
ACCESS_L2 = [
    'No institutional entitlement',
    'Institutional sign-in not working',
    'Wrong account / unexpected account switching',
    'Institution changed (moving universities)',
    'Merge/transfer accounts',
    'Account recovery',
    'Access blocked by security/network controls',
    'Update email address',
    'Registration failed',
    'Password',
    'Close / delete account',
]
UNKNOWN_ACCESS_L2 = 'Unknown rootcause'      # outside the 11, reported separately

# -- Stage 1 ---------------------------------------------------------------
MOD_L1    = 'scopus_l1'
PREFIX_L1 = 'SCLV'
STUB_L1   = 'scl1'

# -- Stage 2: one module per Level 1 with a Level 2 list ---------------------
# (slug, taxonomy prefix, field stub, [Level 1 labels routed to it])
# Prefixes are four letters with no digits: codes are PREFIX + '3.1', and the
# pipeline parses the prefix with a greedy ^[A-Z]+.
L2_MODULES = [
    ('access',            'SCAC', 'scac', ['Access', 'Scopus AI Access']),
    ('performance',       'SCPF', 'scpf', ['Performance & Functionality Issues']),
    ('author_profile',    'SCAP', 'scap', ['Author Profile Correction']),
    ('citation_metrics',  'SCCM', 'sccm', ['Citation / Metrics']),
    ('unclear',           'SCUQ', 'scuq', ['Unclear / No Query']),
    ('funding',           'SCFG', 'scfg', ['Funding / Awarded Grants Data']),
    ('content',           'SCCI', 'scci', ['Content / Indexing']),
    ('help_article',      'SCHA', 'scha', ['Help Article Feedback']),
    ('search_feature',    'SCSF', 'scsf', ['Search / Feature Question']),
    ('institution_admin', 'SCIA', 'scia', ['Institution Admin / Support']),
    ('subscription',      'SCSS', 'scss', ['Subscription / Sales']),
    ('non_scopus',        'SCNS', 'scns', ['Non-Scopus / Wrong Product']),
    ('orcid',             'SCOR', 'scor', ['ORCID Issue']),
    ('api',               'SCAI', 'scai', ['API / Integration']),
]

# Level 1 categories with no Level 2 list: Level 2 assigned by rule, no LLM call.
DETERMINISTIC_L2 = {'Data / Privacy Request': 'Data / Privacy Request'}

MODULE_PREFIXES = {MOD_L1: PREFIX_L1}
MODULE_PREFIXES.update({f'scopus_l2_{slug}': pfx for slug, pfx, _s, _l in L2_MODULES})
ALL_MODULES = list(MODULE_PREFIXES)

_routed = [l1 for *_x, l1s in L2_MODULES for l1 in l1s] + list(DETERMINISTIC_L2)
assert sorted(_routed) == sorted(L1_ORDER), 'every Level 1 needs exactly one route'

# -- Preprocessing parameters (folded into the stage-1 fingerprint) --------
SHORT_CHAR_THRESHOLD = 40      # conversation payload chars below which a ticket may be rule-assigned
TRUNC_CAP            = 16_000  # hard ceiling on any conversation after truncation
TRUNC_HEAD           = 4_000   # kept from the start (the customer's original ask)
TRUNC_TAIL           = 11_900  # kept from the end (where the resolution lives)
assert TRUNC_HEAD + TRUNC_TAIL + 64 <= TRUNC_CAP

# -- Outputs ---------------------------------------------------------------
STEM         = 'scopus26'      # every stage input is named scopus26_* (checkpoint namespace)
OUT_DIR      = BASE / 'runs' / 'output'
INTER_DIR    = BASE / 'runs' / 'intermediate'
SMOKE_DIR    = BASE / 'runs' / 'smoke_scopus_2026'
PROFILE_DIR  = BASE / 'profiles'
STAGE1_INPUT = BASE / 'data' / 'Scopus' / f'{STEM}_stage1_input.csv'
SAMPLE_XLSX  = OUT_DIR / 'scopus_2026_sample_review.xlsx'
FINAL_XLSX   = OUT_DIR / 'scopus_2026_l1_l2_classification.xlsx'
FINAL_CSV    = OUT_DIR / 'scopus_2026_l1_l2_classification_full_text.csv'
for _d in (OUT_DIR, INTER_DIR, SMOKE_DIR, PROFILE_DIR):
    _d.mkdir(parents=True, exist_ok=True)

PROFILE_L1 = PROFILE_DIR / 'scopus_2026_l1.json'

# -- Subprocess environment (UTF-8 matters on Windows) ---------------------
SUBPROCESS_ENV = os.environ.copy()
SUBPROCESS_ENV['PYTHONUTF8'] = '1'
SUBPROCESS_ENV['PYTHONIOENCODING'] = 'utf-8'

for _p in (TICKETS_XLSX, RULES_XLSX):
    assert _p.exists(), f'Missing: {_p}'
print('Project root :', BASE)
print('Python       :', sys.executable)
print('pandas       :', pd.__version__)
print(f'Modules      : 1 Level 1 + {len(L2_MODULES)} Level 2 '
      f'+ {len(DETERMINISTIC_L2)} deterministic')
print('Paths OK')

## 0b. Shared helpers

Copied from notebook 07. `norm_label` makes taxonomy labels stable join keys (zero-width and
non-breaking spaces, stray whitespace) without changing their wording.

In [ ]:
_ZERO_WIDTH = dict.fromkeys(map(ord, '​‌‍﻿'))


def norm_label(value) -> str:
    """Normalise a taxonomy label to a stable join key, preserving wording."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ''
    text = str(value)
    if text.strip().lower() in ('', 'nan', 'none'):
        return ''
    text = text.translate(_ZERO_WIDTH)                    # delete before NFKC
    text = unicodedata.normalize('NFKC', text)            # folds U+00A0 -> ' '
    return re.sub(r'[ \t]+', ' ', text).strip()


def norm_cmp(value) -> str:
    return re.sub(r'\s+', ' ', norm_label(value)).casefold()


def one_line(value) -> str:
    """Collapse newlines too (bullet lists in the SME scenario column)."""
    return re.sub(r'\s+', ' ', norm_label(value)).strip()


def join_parts(*parts) -> str:
    """Join (heading, text) pairs, skipping any with empty text."""
    return ' '.join(f'{head}: {body}' if head else body for head, body in parts if body)


def leaf_labels(module: str) -> dict:
    """code -> label for every leaf of a generated taxonomy JSON."""
    tax = json.loads((TAXONOMY_DIR / f'{module}.json').read_text(encoding='utf-8'))
    return {code: leaf['label']
            for node in tax.values()
            for code, leaf in node.get('children', {}).items()}


def read_stage_output(path) -> pd.DataFrame:
    path = Path(path)
    df = pd.read_excel(path, dtype={ID_COL: str}) if path.suffix.lower().startswith('.xls') \
        else pd.read_csv(path, dtype={ID_COL: str})
    df[ID_COL] = df[ID_COL].astype(str).str.strip()
    return df


def run_cmd(cmd, label):
    print(f'\n$ {" ".join(str(c) for c in cmd)}')
    r = subprocess.run(cmd, text=True, encoding='utf-8', errors='replace',
                       capture_output=True, env=SUBPROCESS_ENV)
    print(r.stdout or '', end='')
    print(r.stderr or '', end='')
    if r.returncode != 0:
        raise RuntimeError(f'{label} failed (exit {r.returncode})')


for raw_, want in [('Access​', 'Access'), (' Password ', 'Password'),
                   ('Admin\xa0Console', 'Admin Console'), (None, ''), ('nan', '')]:
    assert norm_label(raw_) == want, (raw_, norm_label(raw_))
print('Helpers defined; norm_label self-test passes.')

## 0c. Safety net — back up the shared config

`rca build build` rebuilds `generated/pipeline_config.json` from its input workbook
only, so the modules currently in the config are dropped. Their outputs are already saved;
the backup goes to a **new** directory so an earlier backup is never overwritten.

In [ ]:
CONFIG_BACKUP.mkdir(parents=True, exist_ok=True)
_live = json.loads((BASE / 'generated' / 'pipeline_config.json').read_text(encoding='utf-8'))
_backed_up = []
for _src in [BASE / 'generated' / 'pipeline_config.json',
             BASE / 'generated' / 'build_manifest.json',
             *[TAXONOMY_DIR / f'{m}.json' for m in _live['modules']]]:
    if _src.exists():
        _dst = CONFIG_BACKUP / _src.name
        if not _dst.exists():                       # never clobber an earlier backup
            shutil.copy2(_src, _dst)
        _backed_up.append(_src.name)

print(f'Backup dir: {CONFIG_BACKUP.relative_to(BASE)}  ({len(_backed_up)} files)')
print(f'Modules in the live config right now: {list(_live["modules"])}')
(CONFIG_BACKUP / 'RESTORE.txt').write_text(
    'Before notebooks/20_scopus_2026_two_stage.ipynb rebuilt it, generated/pipeline_config.json\n'
    f'held these modules: {list(_live["modules"])}\n\n'
    'To restore, copy pipeline_config.json and build_manifest.json from this folder back into\n'
    'generated/, or rebuild from the definition workbook that produced them, e.g.\n\n'
    '    rca build --input definitions/<that workbook>.xlsx --force-all\n',
    encoding='utf-8')

## 1. Read the rules workbook and apply the revised taxonomy

**Level 1 definitions** are the `How applied` text of the matching row in
`Classification Rules`, verbatim. `Content / Indexing` and `Citation / Metrics` share one row
whose two halves are each tagged with their category; each category gets its own half. `Data / Privacy Request` has no rule row; its definition is the
`Data / Privacy Request` root-cause summary from `Root Cause Descriptions`.

**Decision rules** — the rows that apply across categories — go into the stage-1 prompt
verbatim, followed by an authored **revised-taxonomy override** block that takes precedence
where the original rules mention `Account Management`.

**Level 2 values** come from `Root Cause Descriptions` (summary + SME scenarios). The
`Category` column is sparse in places, so it is forward-filled. Access uses the revised list
of 11 (+ `Unknown rootcause`); the workbook's Scopus AI Access rows are folded into the shared
Access descriptions; the workbook's `Account Management` root causes are folded into the Access
Level 2 values as authored additions.

In [ ]:
# ---- Classification Rules: rule name -> text ------------------------------
_rules = pd.read_excel(RULES_XLSX, sheet_name=SHEET_RULES, dtype=str)
_rules.columns = [norm_label(c) for c in _rules.columns]
RULES = {norm_label(r['Rule / decision']): norm_label(r['How applied'])
         for _, r in _rules.iterrows() if norm_label(r['Rule / decision'])}
print(f'{len(RULES)} rule rows')

# ---- Root Cause Descriptions ---------------------------------------------
_rc = pd.read_excel(RULES_XLSX, sheet_name=SHEET_RC, dtype=str)
_rc.columns = [norm_label(c) for c in _rc.columns]
_rc = _rc.rename(columns={
    'Category': 'category', 'Root Cause': 'l2',
    'Root Cause Summary (Why did this happen?)': 'summary',
    'SME scenarios (Access and Scopus AI Access)': 'scenarios'})
_rc['category'] = _rc['category'].map(norm_label).replace('', pd.NA).ffill()
_rc['l2'] = _rc['l2'].map(norm_label)
_rc['summary'] = _rc['summary'].map(one_line)
_rc['scenarios'] = _rc['scenarios'].fillna('').map(
    lambda s: '; '.join(one_line(x).lstrip('•').strip() for x in str(s).split('\n') if one_line(x)))
_rc = _rc[_rc['l2'] != ''].copy()
RC_ROWS = {(norm_cmp(r['category']), norm_cmp(r['l2'])): r for _, r in _rc.iterrows()}
SHEET_L2 = {}
for _, r in _rc.iterrows():
    SHEET_L2.setdefault(r['category'], []).append(r['l2'])
print(f'{len(_rc)} root-cause rows across {len(SHEET_L2)} categories: {list(SHEET_L2)}')

# ══ Level 1 definitions ═══════════════════════════════════════════════════
L1_RULE_ROW = {
    'Access':                              'Access — what qualifies',
    'Scopus AI Access':                    'Scopus AI Access',
    'Performance & Functionality Issues':  'Performance & Functionality Issues',
    'Author Profile Correction':           'Author Profile Correction',
    'Citation / Metrics':                  'Content / Indexing; Citation / Metrics',
    'Unclear / No Query':                  'Unclear / No Query',
    'Funding / Awarded Grants Data':       'Funding / Awarded Grants Data',
    'Content / Indexing':                  'Content / Indexing; Citation / Metrics',
    'Help Article Feedback':               'Help Article Feedback',
    'Search / Feature Question':           'Search / Feature Question',
    'Institution Admin / Support':         'Institution Admin / Support',
    'Subscription / Sales':                'Subscription / Sales',
    'Non-Scopus / Wrong Product':          'Non-Scopus / Wrong Product',
    'ORCID Issue':                         'ORCID Issue',
    'API / Integration':                   'API / Integration',
}
_missing_rows = [v for v in L1_RULE_ROW.values() if v not in RULES]
assert not _missing_rows, f'rule rows not found in the workbook: {_missing_rows}'

# ── Authored: revised-taxonomy additions (not in the workbook) ─────────────
ACCESS_ADDENDUM = (
    'Account Management is no longer a separate category. Account changes are '
    'Access even when the customer can already sign in: changing or correcting the account email, '
    'account name or details, duplicate or multiple accounts and merging them, closing or '
    'deleting the account, how to register or set up an account, which email is registered or '
    'what the account status is, and password or security-settings questions.')
DATA_PRIVACY_DEF = RC_ROWS[(norm_cmp('Account Management'), norm_cmp('Data / Privacy Request'))]['summary']
DATA_PRIVACY_ADDENDUM = (
    'Use for requests about personal data: what personal data is held, a copy or export of it, '
    'or removal of personal data on privacy or GDPR grounds. A request to close or delete the '
    'account itself is Access; a request to remove an author profile is Author Profile Correction.')

# The workbook gives Content / Indexing and Citation / Metrics ONE shared row, each half
# tagged with its category: "<content signals> (Content / Indexing); <metric signals>
# (Citation / Metrics)." Handing both categories the whole row would give them identical
# definitions, so each gets its own half (verbatim slices, asserted below).
_shared = RULES['Content / Indexing; Citation / Metrics']
_content_part, _citation_part = [x.strip() for x in _shared.split(';', 1)]
assert _content_part.endswith('(Content / Indexing)'), _content_part
assert _citation_part.rstrip('.').endswith('(Citation / Metrics)'), _citation_part
SPLIT_DEFS = {'Content / Indexing': _content_part, 'Citation / Metrics': _citation_part}

L1_DEFS = {}
for l1 in L1_ORDER:
    if l1 == 'Data / Privacy Request':
        L1_DEFS[l1] = join_parts(('Definition', DATA_PRIVACY_DEF),
                                 ('Revised taxonomy', DATA_PRIVACY_ADDENDUM))
    elif l1 == 'Access':
        L1_DEFS[l1] = join_parts(('Definition', RULES[L1_RULE_ROW[l1]]),
                                 ('Revised taxonomy', ACCESS_ADDENDUM))
    elif l1 in SPLIT_DEFS:
        L1_DEFS[l1] = join_parts(('Definition', SPLIT_DEFS[l1]))
    else:
        L1_DEFS[l1] = join_parts(('Definition', RULES[L1_RULE_ROW[l1]]))

# ══ Stage-1 decision rules: verbatim from the workbook, in sheet order ═════
GENERAL_RULE_ROWS = [
    'Primary test — can the customer access Scopus?',
    'Rule 1 — can log in, wants a change',
    'Rule 2 — cannot log in, wants a change to regain access',
    'Rules applied to every category',
    'IP and IPW requests',
    'Access vs Performance & Functionality Issues',
    'Wrong-profile login',
    'Registration and new accounts',
    'Profile views with limited access',
    'Primary issue wins',
    'Empty customer text / chat transcript only',
]
assert all(r in RULES for r in GENERAL_RULE_ROWS), \
    [r for r in GENERAL_RULE_ROWS if r not in RULES]
_used = set(GENERAL_RULE_ROWS) | set(L1_RULE_ROW.values()) | {'Account Management'}
_unused = [r for r in RULES if r not in _used and r != 'Rule / decision']
assert not _unused, f'rule rows that reach no prompt: {_unused}'

GENERAL_RULES_TEXT = ' '.join(f'[{r}] {RULES[r]}' for r in GENERAL_RULE_ROWS)

# ── Authored: overrides for the revised taxonomy (take precedence) ─────────
REVISED_OVERRIDES = (
    'REVISED TAXONOMY OVERRIDES - these take precedence over the rules above. '
    '(1) The Account Management category no longer exists. Wherever a rule above says Account '
    'Management, use Access for account changes (change or correct the account email, account '
    'name or details, duplicate or multiple accounts, merging accounts, closing or deleting the '
    'account, registration or account-setup guidance, registered-email lookup or account status, '
    'password or security-settings questions), even when the customer can sign in, and use '
    'Data / Privacy Request for requests about personal data (what is held, a copy, or removal on '
    'privacy grounds). '
    '(2) Rule 1 still applies to everything else: a customer who can sign in and wants an author '
    'profile, ORCID, content, metrics or institution change is classified by that request, not '
    'as Access. '
    '(3) Scopus AI Access takes precedence over Access whenever the access problem is specifically '
    'about Scopus AI. '
    # Added after the 300-ticket sample: the workbook's empty-customer-text rule was in the
    # prompt verbatim, yet agent-only chats with subjects like 'acceso' or 'unable to find
    # organisation' went to Unclear / No Query.
    '(4) When the ticket has no customer message (for example an agent-only chat transcript), '
    'read the Subject line before choosing Unclear / No Query. If the Subject, in any language, '
    'names a sign-in, login, account or access problem (for example access, acceso, login, '
    'cannot sign in, unable to find organisation, account), classify the ticket as Access. Use '
    'Unclear / No Query only when neither the Subject nor the agent text shows any issue.')

# ══ Level 2 per module ════════════════════════════════════════════════════
# ── Authored: where each former Account Management root cause now lives ─────
ACCESS_L2_ADDENDA = {
    'Update email address': ('Also, revised taxonomy: the customer can sign in but wants the '
                             'account email changed or corrected, or other account name or '
                             'details updated.'),
    'Merge/transfer accounts': ('Also, revised taxonomy: duplicate or multiple accounts that the '
                                'customer wants merged or tidied, even when they can sign in.'),
    'Close / delete account': ('Also, revised taxonomy: a request to delete the account from a '
                               'customer who can sign in.'),
    'Registration failed': ('Also, revised taxonomy: how-to or guidance questions about '
                            'registering or setting up an account, with no blocker.'),
    'Account recovery': ('Also, revised taxonomy: the customer does not know which email is '
                         'registered, or asks what the account status is.'),
    'Password': ('Also, revised taxonomy: questions about password or security settings.'),
}


def access_l2_description(l2: str) -> str:
    row = RC_ROWS.get((norm_cmp('Access'), norm_cmp(l2)))
    assert row is not None, f'Access Level 2 {l2!r} is not in Root Cause Descriptions'
    ai = RC_ROWS.get((norm_cmp('Scopus AI Access'), norm_cmp(l2)))
    return join_parts(('Definition', row['summary']),
                      ('SME scenarios', row['scenarios']),
                      ('For Scopus AI Access', ai['summary'] if ai is not None else ''),
                      ('', ACCESS_L2_ADDENDA.get(l2, '')))


# ── Authored: the smoke test sent agent-only chat transcripts to 'Resolved / Thank-you
#    Only'. By the workbook's own definitions they are 'Empty Chat / No Content': the
#    agent's closing boilerplate is not the customer confirming resolution.
UNCLEAR_L2_ADDENDA = {
    'Empty Chat / No Content': (
        'Also: a chat transcript email that contains only agent text (for example the agent '
        'sending the transcript of our chat, or saying the chat disconnected) and no customer '
        'message stating an issue. Agent boilerplate, greetings and sign-offs do not count as a '
        'customer message.'),
    'Resolved / Thank-you Only': (
        'Use only when the CUSTOMER writes to confirm resolution or to say thanks. An agent '
        'thanking the customer or closing the chat is not this value.'),
}


def sheet_l2_description(category: str, l2: str) -> str:
    row = RC_ROWS[(norm_cmp(category), norm_cmp(l2))]
    extra = UNCLEAR_L2_ADDENDA.get(l2, '') if category == 'Unclear / No Query' else ''
    return join_parts(('Definition', row['summary']), ('SME scenarios', row['scenarios']),
                      ('', extra))


# ── Authored: Access Level 2 precedence. Added after the 300-ticket sample, where
#    'No institutional entitlement' took 50 of 105 Access tickets, including tickets
#    whose cause was never stated and customers who said their institution subscribes.
ACCESS_L2_PRECEDENCE = (
    'Decide in this order. First, if the ticket states a specific account-level cause, choose '
    'that value: the account email is wrong, closed or no longer accessible (Update email '
    'address), the customer moved or left an institution (Institution changed), password, '
    'account recovery or verification codes, duplicate accounts to merge, a security or network '
    'block, registration, landing in the wrong account, or a request to delete the account. '
    'Second, only if the problem is about getting in through the institution, choose between '
    'Institutional sign-in not working (the customer says the institution has Scopus but SSO, '
    'organisation lookup, IP or off-campus access fails) and No institutional entitlement (the '
    'ticket shows the institution does not subscribe, the subscription or trial expired or was '
    'not applied, or an admin must add the user or an IP to grant access). Seeing Scopus Preview '
    'or limited access is a symptom, not a cause. Third, if the ticket never establishes a cause '
    '(only a subject, an acknowledgement, a chat transcript or an unanswered request for '
    'details), choose Unknown rootcause rather than guessing.')

# module slug -> [(label, description)] in code order
L2_TAXONOMY = {}
for slug, _pfx, _stub, l1s in L2_MODULES:
    if slug == 'access':
        L2_TAXONOMY[slug] = ([(l2, access_l2_description(l2)) for l2 in ACCESS_L2]
                             + [(UNKNOWN_ACCESS_L2, access_l2_description(UNKNOWN_ACCESS_L2))])
    else:
        (cat,) = l1s
        assert cat in SHEET_L2, f'{cat!r} has no rows in Root Cause Descriptions'
        L2_TAXONOMY[slug] = [(l2, sheet_l2_description(cat, l2)) for l2 in SHEET_L2[cat]]

# L1 label -> allowed L2 labels (used to validate every result)
L1_TO_L2 = {l1: [lbl for lbl, _d in L2_TAXONOMY[slug]]
            for slug, _p, _s, l1s in L2_MODULES for l1 in l1s}
L1_TO_L2.update({l1: [l2] for l1, l2 in DETERMINISTIC_L2.items()})

# ---- Integrity ----------------------------------------------------------
assert len(L1_ORDER) == len(set(L1_ORDER)) == 16
assert all(L1_DEFS[l1] for l1 in L1_ORDER)
assert len(set(L1_DEFS.values())) == len(L1_DEFS), 'two Level 1 categories share a definition'
assert len(L2_TAXONOMY['access']) == 12
assert set(UNCLEAR_L2_ADDENDA) <= set(SHEET_L2['Unclear / No Query'])
for slug, items in L2_TAXONOMY.items():
    labels = [l for l, _ in items]
    assert len(labels) == len(set(labels)), f'{slug}: duplicate Level 2 labels'
    assert all(d for _, d in items), f'{slug}: a Level 2 value has no description'

# Workbook categories that are intentionally not used as Level 1
_sheet_only = sorted(set(SHEET_L2) - set(L1_ORDER))
assert _sheet_only == ['Account Management'], _sheet_only
_ai_extra = sorted(set(SHEET_L2['Scopus AI Access']) - set(ACCESS_L2))
assert not _ai_extra, f'Scopus AI Access rows outside the Access list: {_ai_extra}'
_acc_sheet = set(SHEET_L2['Access']) - {UNKNOWN_ACCESS_L2}
assert _acc_sheet == set(ACCESS_L2), (
    f'revised Access list vs workbook\n  only revised : {set(ACCESS_L2) - _acc_sheet}'
    f'\n  only workbook: {_acc_sheet - set(ACCESS_L2)}')

TAXONOMY_NOTES = [
    'Account Management (workbook) folded into Access / Data / Privacy Request per the revised '
    'Level 1 list; its root causes are mapped onto Access Level 2 as authored additions.',
    'Scopus AI Access uses all 11 Access Level 2 values (+ Unknown rootcause), per the rules '
    'sheet; the root-cause sheet lists only 4 for it.',
    'Content / Indexing has "Awarded Grant Add Request" and Funding / Awarded Grants Data has '
    '"Missing Awarded Grant": both kept verbatim from the workbook; Level 1 decides which applies.',
]

print(f'\n{"Level 1":<38} Level 2 values')
for l1 in L1_ORDER:
    n = len(L1_TO_L2[l1])
    tag = '(rule, no LLM)' if l1 in DETERMINISTIC_L2 else f'{n}'
    print(f'  {l1:<36} {tag}')
for n in TAXONOMY_NOTES:
    print(f'  note: {n}')
print('\nTaxonomy loaded.')

## 2. Write the pipeline definition workbook

`rca build` reads three sheets — `modules`, `fields`, `taxonomy` — and assigns
taxonomy codes by row order. Every taxonomy is two-level because the generated prompt always
says *"Always use LEAF-level codes"*: in stage 1 each Level 1 is an L1 group with a single leaf
repeating its label, and the guidance sits on the leaf only.

Each module produces three fields: the code, a short justification, and a verbatim evidence
quote. Field stubs avoid the `contact_reason_` / `root_cause_` prefixes that
`drop_enrichment_cols()` strips.

In [ ]:
STAGE1_TASK = (
    'Classify why the customer contacted Scopus support into exactly one of '
    f'{len(L1_ORDER)} Level 1 categories. Each category carries a Definition. The ticket text '
    'starts with the ticket Subject, then the conversation. Assign the single best-fitting LEAF '
    f'code (e.g. {PREFIX_L1}3.1). Apply these decision rules: ')

L2_EVIDENCE_RULE = (
    'A specific Level 2 value applies only when the signals in its own definition are actually '
    'present in the ticket. Do not infer a specific cause from a symptom alone. If the group '
    'has an Unknown value and the ticket never establishes the cause, use it rather than '
    'guessing; otherwise choose the closest value.')

# The smoke test returned an empty code for a vague ticket in a group with no Unknown
# value (Funding); validate_enum_cols keeps it blank. Every ticket needs a value.
ALWAYS_ONE_CODE = 'Always return exactly one code from the taxonomy; never leave the code empty.'

JUST_INSTR = (
    'In one or two sentences, explain why this {kind} applies and why the closest alternative '
    'does not. Refer to what the ticket actually shows. Do not use apostrophes or quotation '
    'marks in your answer.')
EV_INSTR = (
    'Copy verbatim the single sentence or phrase from the transcript that most strongly '
    'supports this {kind}. Do not paraphrase, infer, or add words that are not in the '
    'transcript. If no direct evidence exists, return an empty string.')


def build_modules_sheet() -> pd.DataFrame:
    rows = [{
        'module_name': MOD_L1, 'taxonomy_prefix': PREFIX_L1,
        'taxonomy_file': f'{MOD_L1}.json', 'reorder_group': 'A', 'reorder_order': 1,
        'enabled': True,
        'description': STAGE1_TASK + GENERAL_RULES_TEXT + ' ' + REVISED_OVERRIDES,
        'notes': 'Stage 1 of 2. Runs on every ticket.',
    }]
    for order, (slug, pfx, _stub, l1s) in enumerate(L2_MODULES, start=2):
        names = ' or '.join(f"'{l}'" for l in l1s)
        rows.append({
            'module_name': f'scopus_l2_{slug}', 'taxonomy_prefix': pfx,
            'taxonomy_file': f'scopus_l2_{slug}.json', 'reorder_group': 'B',
            'reorder_order': order, 'enabled': True,
            'description': (
                f'Identify the Level 2 reason for Scopus tickets already classified at Level 1 as '
                f'{names}. Each Level 2 value carries a Definition - use these to separate values '
                f'that look similar. The ticket text starts with the Subject. Assign the single '
                f'best-fitting LEAF code (e.g. {pfx}1.1).'
                + (' ' + ACCESS_L2_PRECEDENCE if slug == 'access' else '')),
            'notes': f'Stage 2. Runs only on the {slug} subset ({len(L2_TAXONOMY[slug])} values).',
        })
    return pd.DataFrame(rows)


def build_fields_sheet() -> pd.DataFrame:
    specs = [(MOD_L1, STUB_L1, 'Level 1 category',
              f'Assign the single leaf-level code (e.g. {PREFIX_L1}3.1) for the Level 1 category '
              f'that best matches this ticket. Read each category Definition and the decision '
              f'rules, and match on what the ticket shows, not on surface keywords. Only use '
              f'codes from the taxonomy. ' + ALWAYS_ONE_CODE)]
    for slug, pfx, stub, _l1s in L2_MODULES:
        specs.append((f'scopus_l2_{slug}', stub, 'Level 2 value',
                      f'Assign the single leaf-level code (e.g. {pfx}1.1) for the Level 2 value '
                      f'that best matches this ticket. Match on each Definition, not on surface '
                      f'keywords. Only use codes from the taxonomy. ' + L2_EVIDENCE_RULE
                      + ' ' + ALWAYS_ONE_CODE))
    rows = []
    for module, stub, kind, code_instr in specs:
        base = {'module_name': module, 'in_output': True, 'explode_list': False, 'notes': ''}
        rows.append({**base, 'field_name': f'{stub}_code', 'field_type': 'code',
                     'description': 'Assigned leaf code', 'prompt_instruction': code_instr,
                     'validate_enum': True})
        rows.append({**base, 'field_name': f'{stub}_justification', 'field_type': 'str',
                     'description': 'Why this code was chosen over the nearest alternative',
                     'prompt_instruction': JUST_INSTR.format(kind=kind), 'validate_enum': False})
        rows.append({**base, 'field_name': f'{stub}_evidence', 'field_type': 'str',
                     'description': 'Verbatim supporting quote from the transcript',
                     'prompt_instruction': EV_INSTR.format(kind=kind), 'validate_enum': False})
    return pd.DataFrame(rows)


def build_taxonomy_sheet() -> pd.DataFrame:
    rows = []
    for l1 in L1_ORDER:
        rows.append({'module_name': MOD_L1, 'level': 'L1', 'label': l1,
                     'description': '', 'notes': ''})
        rows.append({'module_name': MOD_L1, 'level': 'L2', 'label': l1,
                     'description': L1_DEFS[l1], 'notes': ''})
    for slug, _pfx, _stub, l1s in L2_MODULES:
        module = f'scopus_l2_{slug}'
        group = ' / '.join(l1s) if len(l1s) > 1 else l1s[0]
        context = ' '.join(f"'{l}' means: {L1_DEFS[l]}" for l in l1s)
        rows.append({'module_name': module, 'level': 'L1', 'label': group,
                     'description': (f'Every ticket in this task has already been classified at '
                                     f'Level 1 as {" or ".join(repr(l) for l in l1s)}. Choose the '
                                     f'Level 2 value from this group only. {context}'),
                     'notes': 'gating context, authored'})
        for l2, desc in L2_TAXONOMY[slug]:
            rows.append({'module_name': module, 'level': 'L2', 'label': l2,
                         'description': desc, 'notes': ''})
    return pd.DataFrame(rows, columns=['module_name', 'level', 'label', 'description', 'notes'])


modules_df, fields_df, taxonomy_df = (build_modules_sheet(), build_fields_sheet(),
                                      build_taxonomy_sheet())

expected = {MOD_L1: (len(L1_ORDER), len(L1_ORDER))}
expected.update({f'scopus_l2_{slug}': (1, len(L2_TAXONOMY[slug])) for slug, *_r in L2_MODULES})
for module, want in expected.items():
    sub = taxonomy_df[taxonomy_df['module_name'] == module]
    got = (int((sub['level'] == 'L1').sum()), int((sub['level'] == 'L2').sum()))
    assert got == want, f'{module}: expected {want} L1/L2, got {got}'
assert not ((taxonomy_df['level'] == 'L2') & (taxonomy_df['description'] == '')).any()
# _clean_df drops rows whose FIRST column starts with L1/L2/level/module_name/...
assert taxonomy_df.columns[0] == 'module_name'
assert taxonomy_df['module_name'].str.startswith('scopus_').all()

with pd.ExcelWriter(DEFN_XLSX, engine='openpyxl') as w:
    modules_df.to_excel(w, sheet_name='modules', index=False)
    fields_df.to_excel(w, sheet_name='fields', index=False)
    taxonomy_df.to_excel(w, sheet_name='taxonomy', index=False)

print(f'Wrote {DEFN_XLSX.relative_to(BASE)}: {len(modules_df)} modules, {len(fields_df)} fields, '
      f'{len(taxonomy_df)} taxonomy rows')
for module, (l1, l2) in expected.items():
    print(f'  {module:<30} {MODULE_PREFIXES[module]:<5} {l1:>2} L1 / {l2:>2} L2')

## 3. Build the pipeline artefacts

Dry run first, then the real build of exactly these modules.

In [ ]:
run_cmd([sys.executable, '-m', 'rca_pipeline.cli', 'build',
         '--input', str(DEFN_XLSX), '--dry-run'], 'build_pipeline --dry-run')

In [ ]:
run_cmd([sys.executable, '-m', 'rca_pipeline.cli', 'build',
         '--input', str(DEFN_XLSX), '--force', *ALL_MODULES], 'build_pipeline')

cfg = json.loads((BASE / 'generated' / 'pipeline_config.json').read_text(encoding='utf-8'))
assert set(cfg['modules']) == set(ALL_MODULES), (
    f'config modules mismatch\n  missing: {set(ALL_MODULES) - set(cfg["modules"])}'
    f'\n  extra  : {set(cfg["modules"]) - set(ALL_MODULES)}')
for module, (_l1, want_leaves) in expected.items():
    assert (MODULES_DIR / f'{module}.py').exists(), f'missing {module}.py'
    assert len(leaf_labels(module)) == want_leaves, module
    assert cfg['modules'][module]['taxonomy_prefix'] == MODULE_PREFIXES[module]

from rca_pipeline.utils import _STRUCTURAL_PREFIXES
for stub in [STUB_L1] + [s for _sl, _p, s, _l in L2_MODULES]:
    for bad in _STRUCTURAL_PREFIXES:
        assert not f'{stub}_'.startswith(bad), f'field stub {stub}_ collides with {bad!r}'
_pfx = list(MODULE_PREFIXES.values())
assert all(re.fullmatch(r'[A-Z]+', p) for p in _pfx)
assert all(a == b or not b.startswith(a) for a in _pfx for b in _pfx)

L1_LABELS = leaf_labels(MOD_L1)                    # code -> Level 1 label
LABEL_TO_L1_CODE = {v: k for k, v in L1_LABELS.items()}
assert set(LABEL_TO_L1_CODE) == set(L1_ORDER)
print(f'\nBuild verified: {len(ALL_MODULES)} modules, '
      f'{sum(len(leaf_labels(m)) for m in ALL_MODULES)} leaves.\n')
for c, l in L1_LABELS.items():
    print(f'  {c:<9} {l}')

## 4. Gate 1 — review the exact prompts *(free, read this output)*

Renders every system prompt exactly as the model receives it and asserts that every
definition survives into it verbatim. Then fingerprints each prompt for the checkpoint guard.

In [ ]:
import importlib

from rca_pipeline.taxonomy import taxonomy_to_rich_prompt

importlib.invalidate_caches()
prompts = {}
for module in ALL_MODULES:
    tax = json.loads((TAXONOMY_DIR / f'{module}.json').read_text(encoding='utf-8'))
    name = f'generated.modules.{module}'
    mod = importlib.reload(sys.modules[name]) if name in sys.modules \
        else importlib.import_module(name)
    prompts[module] = mod._build_system_prompt(
        taxonomy_to_rich_prompt(tax, MODULE_PREFIXES[module]))

checked = 0
for l1 in L1_ORDER:
    assert l1 in prompts[MOD_L1], f'Level 1 label missing: {l1!r}'
    assert L1_DEFS[l1] in prompts[MOD_L1], f'Level 1 definition missing: {l1!r}'
    checked += 2
for r in GENERAL_RULE_ROWS:
    assert RULES[r] in prompts[MOD_L1], f'decision rule missing: {r!r}'
    checked += 1
assert REVISED_OVERRIDES in prompts[MOD_L1]
assert all(ALWAYS_ONE_CODE in p for p in prompts.values())
assert ACCESS_L2_PRECEDENCE in prompts['scopus_l2_access']
for slug, *_r in L2_MODULES:
    p = prompts[f'scopus_l2_{slug}']
    assert L2_EVIDENCE_RULE in p, f'evidence rule missing: {slug}'
    for l2, desc in L2_TAXONOMY[slug]:
        assert l2 in p and desc in p, f'{slug}: {l2!r} or its definition missing'
        checked += 2
print(f'Verbatim check: all {checked} taxonomy strings present in the prompts.\n')

PROMPT_FINGERPRINTS = {m: hashlib.sha256(p.encode('utf-8')).hexdigest()[:16]
                       for m, p in prompts.items()}
_preproc = f'{SHORT_CHAR_THRESHOLD}|{TRUNC_CAP}|{TRUNC_HEAD}|{TRUNC_TAIL}|subject-prefix'
for m in ALL_MODULES:      # every stage sees the preprocessed text
    PROMPT_FINGERPRINTS[m] = hashlib.sha256(
        (prompts[m] + _preproc).encode('utf-8')).hexdigest()[:16]

print(f'{"module":<30} {"chars":>7} {"~tokens":>8}  fingerprint')
for m in ALL_MODULES:
    print(f'{m:<30} {len(prompts[m]):>7,} {len(prompts[m]) // 4:>8,}  {PROMPT_FINGERPRINTS[m]}')

In [ ]:
print('=' * 78)
print('FULL STAGE 1 SYSTEM PROMPT')
print('=' * 78)
print(prompts[MOD_L1])

In [ ]:
# Every stage-2 prompt, in full. Long, but this is the one chance to read them for free.
for slug, *_r in L2_MODULES:
    print('=' * 78)
    print(f'scopus_l2_{slug}')
    print('=' * 78)
    print(prompts[f'scopus_l2_{slug}'])
    print()

## 5. Gate 2 — preprocess the tickets *(free)*

**Scope.** Every row of the file is in scope; the year of `incident_closed_datetime` is
asserted to be 2026 rather than filtered.

**Subject prefix.** The rules say that when the customer text is empty the subject and agent
text decide the category, so the model sees `Subject: <incident_subject>` followed by the
conversation.

**Empty-ticket rule.** A ticket whose conversation payload (speaker markers stripped) is under
`SHORT_CHAR_THRESHOLD` characters **and** whose subject is empty is assigned
*Unclear / No Query → Empty Chat / No Content* with no LLM call. Short tickets that do have a
subject still go to the model: the rules say a subject pointing at sign-in keeps the ticket in
Access.

**Truncation** to a 16,000-char ceiling — first 4,000 + last 11,900 with a visible marker —
bounds the worst-case request. The original text is kept for the deliverable.

In [ ]:
raw = pd.read_excel(TICKETS_XLSX, dtype=str)
raw[ID_COL] = raw[ID_COL].astype(str).str.strip()
print(f'{TICKETS_XLSX.name}: {len(raw):,} rows x {len(raw.columns)} cols')
assert raw[ID_COL].is_unique, 'duplicate INCIDENT_REF in the source file'

raw['closed_date'] = pd.to_datetime(raw[DATE_COL], errors='coerce')
assert raw['closed_date'].notna().all(), 'unparseable incident_closed_datetime'
assert (raw['closed_date'].dt.year == TARGET_YEAR).all(), 'tickets closed outside 2026'
tickets = raw.copy()
tickets['month'] = tickets['closed_date'].dt.to_period('M').astype(str)
print(tickets['month'].value_counts().sort_index().to_string())

_MARKER_RE = re.compile(r'(?mi)^\s*(customer|agent)(\s+notes)?\s*:')


def payload_len(text) -> int:
    return len(_MARKER_RE.sub('', str(text)).strip())


def clean_subject(s) -> str:
    s = norm_label(s)
    return '' if s.casefold() in ('no value', 'no subject') else s


tickets[TEXT_COL] = tickets[TEXT_COL].fillna('')
tickets['subject_clean'] = tickets[SUBJECT_COL].map(clean_subject)
tickets['conversation_chars_original'] = tickets[TEXT_COL].str.len()
tickets['payload_chars'] = tickets[TEXT_COL].map(payload_len)
tickets['excel_cell_capped'] = tickets['conversation_chars_original'] >= 32_767

# Characters written by the customer (text after 'Customer:' markers). Zero means an
# agent-only transcript; used by the Resolved/Thank-you correction in stage 2.
_CUST_RE = re.compile(r'(?mi)^\s*customer\s*:(.*)$')
tickets['customer_chars'] = tickets[TEXT_COL].map(
    lambda x: sum(len(m.strip()) for m in _CUST_RE.findall(str(x))))
NO_CUSTOMER_TEXT = set(tickets.loc[tickets['customer_chars'] == 0, ID_COL])
print(f'Tickets with no customer text at all: {len(NO_CUSTOMER_TEXT):,}')

is_empty = (tickets['payload_chars'] < SHORT_CHAR_THRESHOLD) & (tickets['subject_clean'] == '')
rule_assigned = tickets[is_empty].copy()
to_classify = tickets[~is_empty].copy().reset_index(drop=True)
print(f'\nShort conversations (payload < {SHORT_CHAR_THRESHOLD}): '
      f'{int((tickets["payload_chars"] < SHORT_CHAR_THRESHOLD).sum())}, of which '
      f'{len(rule_assigned)} also have no subject -> rule-assigned')
print(f'Conversations already cut at the Excel cell limit in the source: '
      f'{int(tickets["excel_cell_capped"].sum())}')

TRUNC_MARKER = '\n\n[...TRUNCATED {n:,} CHARS FROM MIDDLE...]\n\n'


def truncate_middle(text) -> str:
    s = str(text)
    if len(s) <= TRUNC_CAP:
        return s
    cut = len(s) - TRUNC_HEAD - TRUNC_TAIL
    return s[:TRUNC_HEAD] + TRUNC_MARKER.format(n=cut) + s[-TRUNC_TAIL:]


def model_text(row) -> str:
    subj = row['subject_clean'] or '(none)'
    return truncate_middle(f'Subject: {subj}\n\n{row[TEXT_COL]}')


to_classify['model_text'] = to_classify.apply(model_text, axis=1)
to_classify['was_truncated'] = to_classify['model_text'].str.contains(
    r'\[\.\.\.TRUNCATED [\d,]+ CHARS FROM MIDDLE\.\.\.\]', regex=True)
_kept = to_classify['model_text'].str.len().sum()
print(f'Truncated {int(to_classify["was_truncated"].sum())} of {len(to_classify):,}; text kept '
      f'{_kept / to_classify["conversation_chars_original"].sum() * 100:.1f}%')

# The pipeline reads TEXT_COL, so the stage input carries the model text under that name.
STAGE_TEXT = (to_classify[[ID_COL, 'model_text']]
              .rename(columns={'model_text': TEXT_COL}).reset_index(drop=True))
STAGE_TEXT.to_csv(STAGE1_INPUT, index=False)

assert len(tickets) == len(to_classify) + len(rule_assigned)
assert STAGE_TEXT[TEXT_COL].str.len().max() <= TRUNC_CAP
assert STAGE_TEXT[ID_COL].is_unique
print(f'\n{len(tickets):,} = {len(to_classify):,} to classify + {len(rule_assigned)} '
      f'rule-assigned   OK')
print(f'Wrote {STAGE1_INPUT.relative_to(BASE)}')

## 6. Profiles, routes and the checkpoint guard

`rca run` applies a profile key only when the matching CLI arg is still at its
default, so an unrecognised key would be **silently ignored**. Every key is validated against
the real argparse flags first. Tuning is inherited from notebook 07.

`ROUTES` is keyed by predicted Level 1 **code**. Access and Scopus AI Access both route to the
Access module but get separate subset files (so separate checkpoints).

In [ ]:
_help = subprocess.run([sys.executable, '-m', 'rca_pipeline.cli', 'run', '--help'],
                       text=True, encoding='utf-8', errors='replace',
                       capture_output=True, env=SUBPROCESS_ENV)
assert _help.returncode == 0, f'rca run --help failed:\n{_help.stderr[-2000:]}'
_dests = {f.replace('-', '_') for f in re.findall(r'--([a-z][a-z0-9-]+)', _help.stdout)}
assert {'input', 'output', 'batch_size', 'max_chars', 'modules'} <= _dests

TUNING = {'batch_size': 5, 'max_chars': 40_000, 'concurrency': 4, 'retries': 5,
          'max_tokens': 8192, 'track_costs': True, 'cost_report_dir': 'runs/cost_reports'}


def write_profile(path: Path, *, comment, input_rel, output_rel, modules) -> Path:
    spec = {'_comment': comment, 'input': input_rel, 'output': output_rel,
            'taxonomy_dir': 'generated/taxonomies', 'text_col': TEXT_COL, 'id_col': ID_COL,
            'modules': modules, **TUNING}
    unknown = [k for k in spec if k != '_comment' and k not in _dests]
    assert not unknown, f'{path.name}: keys rca run would silently ignore: {unknown}'
    path.write_text(json.dumps(spec, indent=2) + '\n', encoding='utf-8')
    return path


write_profile(PROFILE_L1,
              comment=('Stage 1 of 2 - Level 1 category for every 2026 Scopus ticket. Input is '
                       'written by notebooks/20_scopus_2026_two_stage.ipynb (subject prefixed, '
                       'empty tickets removed, conversations capped).'),
              input_rel=STAGE1_INPUT.relative_to(BASE).as_posix(),
              output_rel=(OUT_DIR / f'{STEM}_step1_l1.xlsx').relative_to(BASE).as_posix(),
              modules=[MOD_L1])

ROUTES = {}
for slug, pfx, stub, l1s in L2_MODULES:
    module = f'scopus_l2_{slug}'
    profile = write_profile(
        PROFILE_DIR / f'scopus_2026_l2_{slug}.json',
        comment=(f'Stage 2 of 2 - Level 2 for tickets classified as {" / ".join(l1s)}. Input is a '
                 f'subset CSV written by notebooks/20_scopus_2026_two_stage.ipynb.'),
        input_rel=(INTER_DIR / f'{STEM}_subset_{slug}.csv').relative_to(BASE).as_posix(),
        output_rel=(OUT_DIR / f'{STEM}_step2_{slug}.xlsx').relative_to(BASE).as_posix(),
        modules=[module])
    for l1 in l1s:
        sub_slug = slug if len(l1s) == 1 else re.sub(r'[^a-z0-9]+', '_', l1.lower()).strip('_')
        ROUTES[LABEL_TO_L1_CODE[l1]] = {'l1': l1, 'slug': sub_slug, 'module': module,
                                        'stub': stub, 'profile': profile, 'kind': 'llm'}
for l1, l2 in DETERMINISTIC_L2.items():
    ROUTES[LABEL_TO_L1_CODE[l1]] = {'l1': l1, 'slug': '', 'module': '', 'stub': '',
                                    'profile': None, 'kind': 'deterministic', 'l2': l2}
assert set(ROUTES) == set(L1_LABELS), 'every Level 1 code needs a route'
assert len({r['slug'] for r in ROUTES.values() if r['kind'] == 'llm'}) == len(L1_ORDER) - 1

print(f'Wrote {1 + len(L2_MODULES)} profiles; {len(ROUTES)} Level 1 codes routed:')
for c in sorted(ROUTES, key=lambda c: int(re.search(r'\d+', c).group())):
    r = ROUTES[c]
    print(f'  {c:<9} {r["l1"]:<36} {r["module"] or "(rule) -> " + repr(r["l2"])}')

In [ ]:
CHECKPOINT_DIR = BASE / 'runs' / 'checkpoints'


def checkpoint_files(input_path) -> list:
    stem = Path(input_path).stem
    return [CHECKPOINT_DIR / f'{stem}_checkpoint.jsonl', CHECKPOINT_DIR / f'{stem}_bad_uids.txt']


def _fingerprint_file(input_path) -> Path:
    return CHECKPOINT_DIR / f'{Path(input_path).stem}_checkpoint.fingerprint'


def _read_fingerprints(fp: Path) -> dict:
    if not fp.exists():
        return {}
    try:
        stored = json.loads(fp.read_text(encoding='utf-8'))
    except (json.JSONDecodeError, UnicodeDecodeError):
        return {}
    return stored if isinstance(stored, dict) else {}


def _sync_checkpoint(input_path, module: str) -> None:
    """Drop this input's checkpoint if a different prompt produced it."""
    want = PROMPT_FINGERPRINTS[module]
    fp = _fingerprint_file(input_path)
    stored = _read_fingerprints(fp)
    if stored.get(module) == want:
        return
    dropped = [p.name for p in checkpoint_files(input_path) if p.exists()]
    for p in checkpoint_files(input_path):
        p.unlink(missing_ok=True)
    if dropped:
        print(f'  {module}: prompt changed ({stored.get(module) or "no fingerprint"} -> {want}); '
              f'discarded {", ".join(dropped)}')


def _stamp_checkpoint(input_path, module: str) -> None:
    fp = _fingerprint_file(input_path)
    fp.parent.mkdir(parents=True, exist_ok=True)
    stored = _read_fingerprints(fp)
    stored[module] = PROMPT_FINGERPRINTS[module]
    fp.write_text(json.dumps(stored, indent=2), encoding='utf-8')


def run_stage(profile: Path, *, module: str, input_path=None, output_path=None,
              label='') -> Path:
    """Run one pipeline stage as a subprocess; CLI flags override the profile's paths."""
    spec = json.loads(profile.read_text(encoding='utf-8'))
    resolved_in = Path(input_path) if input_path else BASE / spec['input']
    resolved_out = Path(output_path) if output_path else BASE / spec['output']
    resolved_out.parent.mkdir(parents=True, exist_ok=True)
    # Checkpoints are keyed by the input file's STEM only: namespacing every stage
    # input scopus26_* stops it resuming from another notebook's checkpoint.
    assert resolved_in.stem.startswith(f'{STEM}_'), f'{resolved_in.name}: must be {STEM}_*'

    cmd = [sys.executable, '-m', 'rca_pipeline.cli', 'run', '--profile', str(profile),
           '--input', str(resolved_in), '--output', str(resolved_out)]
    banner = label or profile.stem
    print('=' * 78 + f'\nSTAGE: {banner}\n' + '=' * 78)
    _sync_checkpoint(resolved_in, module)
    print(' '.join(str(c) for c in cmd), flush=True)

    err_lines = []

    def _drain(stream):
        for line in iter(stream.readline, ''):
            err_lines.append(line)

    t0 = time.time()
    proc = subprocess.Popen(cmd, text=True, encoding='utf-8', errors='replace',
                            stdout=subprocess.PIPE, stderr=subprocess.PIPE, bufsize=1,
                            env=SUBPROCESS_ENV)
    t = threading.Thread(target=_drain, args=(proc.stderr,), daemon=True)
    t.start()
    for line in iter(proc.stdout.readline, ''):
        print(line, end='', flush=True)
    proc.wait()
    t.join(timeout=5)
    if proc.returncode != 0:
        print(f'\nExit code {proc.returncode}\n--- stderr (last 60 lines) ---')
        print(''.join(err_lines[-60:]))
        raise RuntimeError(f'{banner} failed with exit code {proc.returncode}')
    _stamp_checkpoint(resolved_in, module)
    print(f'\n-> {banner} done in {time.time() - t0:.0f}s  ->  {resolved_out}')
    return resolved_out


def costs_since(t0: float) -> float:
    """Sum total_cost_usd over cost reports written after t0."""
    total = 0.0
    for p in (BASE / 'runs' / 'cost_reports').glob('costs_*.json'):
        if p.stat().st_mtime >= t0:
            try:
                c = json.loads(p.read_text(encoding='utf-8'))
                total += float(c.get('summary', c).get('total_cost_usd') or 0)
            except Exception:
                pass
    return total


print('Checkpoint guard armed.')

### Routing helpers

`run_stage1` turns a pipeline output into `[ID, l1_code, level1, ...]`; `run_stage2_over`
routes tickets by predicted Level 1 code, runs each Level 2 module on its subset, assigns the
deterministic Level 2, and asserts every Level 2 is allowed under its Level 1.

In [ ]:
def run_stage1(input_csv: Path, output_xlsx: Path, label: str) -> pd.DataFrame:
    out = run_stage(PROFILE_L1, module=MOD_L1, input_path=input_csv, output_path=output_xlsx,
                    label=label)
    res = read_stage_output(out)
    gcol = f'{STUB_L1}_evidence_grounded'
    s1 = pd.DataFrame({
        ID_COL: res[ID_COL],
        'level1_code': res[f'{STUB_L1}_code'].astype(str).str.strip(),
        'level1_justification': res[f'{STUB_L1}_justification'],
        'level1_evidence': res[f'{STUB_L1}_evidence'],
        'level1_evidence_grounded': res[gcol] if gcol in res.columns else pd.NA,
    })
    s1['level1'] = s1['level1_code'].map(lambda c: L1_LABELS.get(c, ''))
    n_in = len(pd.read_csv(input_csv, dtype=str, usecols=[ID_COL]))
    assert len(s1) == n_in, f'row count changed ({len(s1)} out, {n_in} in)'
    assert s1[ID_COL].is_unique
    return s1


def run_stage2_over(s1: pd.DataFrame, text_src: pd.DataFrame, *, out_dir: Path, tag: str,
                    label: str) -> pd.DataFrame:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    sizes = {c: int((s1['level1_code'] == c).sum()) for c in ROUTES}
    frames = []
    for code_ in sorted((c for c in ROUTES if ROUTES[c]['kind'] == 'llm'), key=lambda c: -sizes[c]):
        route, n = ROUTES[code_], sizes[code_]
        if n == 0:
            continue
        ids = set(s1.loc[s1['level1_code'] == code_, ID_COL])
        sub = text_src[text_src[ID_COL].isin(ids)].reset_index(drop=True)
        assert len(sub) == n, f'{route["slug"]}: {len(sub)} rows for {n} predicted'
        sub_csv = out_dir / f'{STEM}{tag}_subset_{route["slug"]}.csv'
        sub.to_csv(sub_csv, index=False)
        out = run_stage(route['profile'], module=route['module'], input_path=sub_csv,
                        output_path=out_dir / f'{STEM}{tag}_step2_{route["slug"]}.xlsx',
                        label=f'{label} - {route["slug"]} ({n} tickets)')
        res = read_stage_output(out)
        assert len(res) == n, f'{route["slug"]}: row count changed'
        stub, labels = route['stub'], leaf_labels(route['module'])
        gcol = f'{stub}_evidence_grounded'
        frames.append(pd.DataFrame({
            ID_COL: res[ID_COL],
            'level2_code': res[f'{stub}_code'].astype(str).str.strip(),
            'level2': res[f'{stub}_code'].map(lambda c: labels.get(str(c).strip(), '')),
            'level2_justification': res[f'{stub}_justification'],
            'level2_evidence': res[f'{stub}_evidence'],
            'level2_evidence_grounded': res[gcol] if gcol in res.columns else pd.NA,
            'level2_module': route['module'],
        }))
    for code_, route in ROUTES.items():
        if route['kind'] != 'deterministic' or sizes[code_] == 0:
            continue
        ids = sorted(s1.loc[s1['level1_code'] == code_, ID_COL])
        print(f'  {route["l1"]:<36} {len(ids):>5} tickets -> {route["l2"]!r} by rule')
        frames.append(pd.DataFrame({
            ID_COL: ids, 'level2_code': '', 'level2': route['l2'],
            'level2_justification': 'This Level 1 category has no Level 2 list.',
            'level2_evidence': '', 'level2_evidence_grounded': pd.NA, 'level2_module': ''}))
    l2 = pd.concat(frames, ignore_index=True)
    assert l2[ID_COL].is_unique, 'a ticket received two Level 2 values'

    # Rule: 'Resolved / Thank-you Only' means the CUSTOMER confirmed or thanked. With no
    # customer text that is impossible, and the model was seen picking it for agent-only
    # chat transcripts despite the guidance, so those become 'Empty Chat / No Content'.
    l2['level2_override'] = ''
    fix = (l2['level2'] == 'Resolved / Thank-you Only') & l2[ID_COL].isin(NO_CUSTOMER_TEXT)
    if fix.any():
        l2.loc[fix, 'level2_override'] = 'rule:no_customer_text (model said Resolved / Thank-you Only)'
        l2.loc[fix, 'level2'] = 'Empty Chat / No Content'
        l2.loc[fix, 'level2_code'] = {v: k for k, v in leaf_labels('scopus_l2_unclear').items()}[
            'Empty Chat / No Content']
        print(f'  Resolved/Thank-you -> Empty Chat (no customer text): {int(fix.sum())}')

    chk = l2.merge(s1[[ID_COL, 'level1']], on=ID_COL)
    bad = chk[[b not in L1_TO_L2.get(a, []) for a, b in zip(chk['level1'], chk['level2'])]]
    print(f'Level 2 blank/invalid: {len(bad)} of {len(chk)}')
    if len(bad):
        print(bad[[ID_COL, 'level1', 'level2_code', 'level2']].head(20).to_string(index=False))
    return l2


print('Routing helpers defined.')

## 7. Gate 3 — 30-ticket smoke test *(~$0.30)*

30 tickets spread across the Oracle contact reasons, both stages. Asserts: no blank or
invalid Level 1 codes, every Level 2 allowed under its Level 1. Then prints each prediction
next to the subject and Oracle tag so it can be read by hand, and the measured cost.

In [ ]:
SMOKE_N = 30
_oracle = to_classify[[ID_COL, ORACLE_L1]].copy()
_per = max(1, SMOKE_N // _oracle[ORACLE_L1].nunique())
smoke_ids = (_oracle.sample(frac=1, random_state=0).groupby(ORACLE_L1).head(_per)[ID_COL]
             .tolist())
smoke_ids += [i for i in _oracle.sample(frac=1, random_state=1)[ID_COL] if i not in smoke_ids]
smoke_ids = smoke_ids[:SMOKE_N]
smoke_text = STAGE_TEXT[STAGE_TEXT[ID_COL].isin(smoke_ids)].reset_index(drop=True)
smoke_in = SMOKE_DIR / f'{STEM}_smoke_input.csv'
smoke_text.to_csv(smoke_in, index=False)
print(f'{len(smoke_text)} tickets; Oracle contact reasons:')
print(to_classify.loc[to_classify[ID_COL].isin(smoke_ids), ORACLE_L1].value_counts().to_string())

_t0 = time.time()
smoke_s1 = run_stage1(smoke_in, SMOKE_DIR / 'smoke_step1.xlsx', f'SMOKE stage 1 - {len(smoke_text)}')
_blank = int((smoke_s1['level1'] == '').sum())
print(f'\nBlank/invalid Level 1 codes: {_blank}')
assert _blank == 0, 'stage 1 produced codes outside the taxonomy - fix before spending more'
print(smoke_s1['level1'].value_counts().to_string())

In [ ]:
smoke_s2 = run_stage2_over(smoke_s1, smoke_text, out_dir=SMOKE_DIR, tag='_smoke',
                           label='SMOKE stage 2')
_chk = smoke_s2.merge(smoke_s1[[ID_COL, 'level1']], on=ID_COL)
assert len(smoke_s2) == len(smoke_s1), 'a smoke ticket got no Level 2 row'
# A label outside its Level 1 list is impossible by construction and must never happen.
# A BLANK code is the one real failure mode: the model abstains on a content-free ticket
# routed to a group with no Unknown value. Tolerated at a low rate, reported, never hidden.
_nonblank_bad = _chk[(_chk['level2'] != '')
                     & ~pd.Series([b in L1_TO_L2[a] for a, b in zip(_chk['level1'], _chk['level2'])],
                                  index=_chk.index)]
assert _nonblank_bad.empty, f'smoke: Level 2 outside its Level 1 list:\n{_nonblank_bad}'
_l2_blank = int((_chk['level2'] == '').sum())
print(f'Blank Level 2: {_l2_blank} of {len(_chk)}')
assert _l2_blank <= max(1, round(0.02 * len(_chk))), 'too many blank Level 2 codes'

SMOKE_COST = costs_since(_t0)    # 0 when every stage resumed from a checkpoint
print(f'\nSmoke cost: ${SMOKE_COST:.4f} for {len(smoke_s1)} tickets '
      f'(${SMOKE_COST / len(smoke_s1):.5f}/ticket, both stages)')
print(f'Projected full run on {len(to_classify):,} tickets: '
      f'${SMOKE_COST / len(smoke_s1) * len(to_classify):,.0f}')
_g = pd.concat([smoke_s1['level1_evidence_grounded'], smoke_s2['level2_evidence_grounded']])
print(f'Evidence quotes not found in transcript: {int((_g == False).sum())} of {int(_g.notna().sum())}')

In [ ]:
# Read these against the conversations: is the Level 1/2 what a human would pick?
smoke_view = (smoke_s1.merge(smoke_s2, on=ID_COL)
              .merge(tickets[[ID_COL, SUBJECT_COL, ORACLE_L1, ORACLE_L2]], on=ID_COL))
with pd.option_context('display.max_colwidth', 70, 'display.width', 250):
    print(smoke_view[[ID_COL, SUBJECT_COL, ORACLE_L1, 'level1', 'level2']]
          .assign(**{SUBJECT_COL: smoke_view[SUBJECT_COL].str.slice(0, 50)}).to_string(index=False))
for _, r in smoke_view.head(8).iterrows():
    print(f'\n--- {r[ID_COL]}  {r["level1"]} / {r["level2"]}')
    print(f'    L1 why : {r["level1_justification"]}')
    print(f'    L2 why : {r["level2_justification"]}')

## 8. Gate 4 — 300-ticket sample for SME review *(~$3)*

No gold labels exist for Scopus, so this gate produces a stratified sample (proportional to
the Oracle contact reason, at least 3 per reason) with blank review columns, plus a crosstab
of predicted Level 1 against the Oracle tag. The crosstab is a sanity check only — the Oracle
tags use a different taxonomy and the rules say they are not evidence.

Review `runs/output/scopus_2026_sample_review.xlsx`, then set `STOP_BEFORE_FULL_RUN = False`.

In [ ]:
SAMPLE_N = 300
_counts = to_classify[ORACLE_L1].value_counts()
_alloc = (_counts / _counts.sum() * SAMPLE_N).round().clip(lower=3).astype(int)
sample_ids = []
for reason, n in _alloc.items():
    pool = to_classify.loc[to_classify[ORACLE_L1] == reason, ID_COL]
    sample_ids += pool.sample(n=min(n, len(pool)), random_state=0).tolist()
sample_text = STAGE_TEXT[STAGE_TEXT[ID_COL].isin(sample_ids)].reset_index(drop=True)
sample_in = SMOKE_DIR / f'{STEM}_sample_input.csv'
sample_text.to_csv(sample_in, index=False)
print(f'Sample: {len(sample_text)} tickets across {len(_alloc)} Oracle contact reasons')

_t0 = time.time()
sample_s1 = run_stage1(sample_in, SMOKE_DIR / 'sample_step1.xlsx', f'SAMPLE stage 1 - {len(sample_text)}')
assert (sample_s1['level1'] == '').mean() <= 0.005, 'blank Level 1 rate above 0.5%'
sample_s2 = run_stage2_over(sample_s1, sample_text, out_dir=SMOKE_DIR, tag='_sample',
                            label='SAMPLE stage 2')
SAMPLE_COST = costs_since(_t0)
print(f'\nSample cost ${SAMPLE_COST:.2f} -> ${SAMPLE_COST / len(sample_s1):.5f}/ticket; projected '
      f'full run ${SAMPLE_COST / len(sample_s1) * len(to_classify):,.0f}')

sample_view = (sample_s1.merge(sample_s2, on=ID_COL, how='left')
               .merge(tickets[[ID_COL, 'Channel', SUBJECT_COL, ORACLE_L1, ORACLE_L2, TEXT_COL]],
                      on=ID_COL))
print('\nPredicted Level 1 on the sample:')
print(sample_view['level1'].value_counts().to_string())
SAMPLE_XTAB = pd.crosstab(sample_view['level1'], sample_view[ORACLE_L1])

review = sample_view[[ID_COL, 'Channel', SUBJECT_COL, ORACLE_L1, ORACLE_L2, 'level1', 'level2',
                      'level2_override',
                      'level1_justification', 'level2_justification', 'level1_evidence',
                      'level2_evidence', TEXT_COL]].copy()
for c in ('L1 correct? (Y/N)', 'Correct L1 if N', 'L2 correct? (Y/N)', 'Correct L2 if N',
          'Reviewer comment'):
    review[c] = ''
review[TEXT_COL] = review[TEXT_COL].str.slice(0, 32_000)
with pd.ExcelWriter(SAMPLE_XLSX, engine='openpyxl') as w:
    review.to_excel(w, sheet_name='Sample for review', index=False)
    SAMPLE_XTAB.to_excel(w, sheet_name='L1 vs Oracle')
print(f'\nWrote {SAMPLE_XLSX.relative_to(BASE)}')

In [ ]:
with pd.option_context('display.width', 250, 'display.max_columns', 40):
    print(SAMPLE_XTAB)

### Decision point

Look at the sample review file and the crosstab. If the categories look right, set
`STOP_BEFORE_FULL_RUN = False` and continue. If a definition needs adjusting, edit section 1/2,
re-run from the top — the fingerprint guard discards only the affected checkpoints.

In [ ]:
STOP_BEFORE_FULL_RUN = True
if STOP_BEFORE_FULL_RUN:
    raise SystemExit('STOP_BEFORE_FULL_RUN is True - review the sample, then set it False and '
                     're-run from here.')

## 9. Stage 1 — Level 1 for every ticket *(the bulk of the cost)*

Resumable: an interruption picks up from the checkpoint.

In [ ]:
step1 = run_stage1(STAGE1_INPUT, OUT_DIR / f'{STEM}_step1_l1.xlsx', 'FULL stage 1 - Level 1')
print(f'\nRows: {len(step1):,}')
for label, n in step1['level1'].replace('', '(blank)').value_counts().items():
    print(f'  {n:>6}  {n / len(step1) * 100:>5.1f}%  {label}')
_blank = step1[step1['level1'] == '']
print(f'\nBlank/invalid: {len(_blank)}')
assert len(_blank) / len(step1) <= 0.005, 'blank rate above 0.5% - investigate'

## 10. Split by Level 1 and run the Level 2 modules

In [ ]:
level2 = run_stage2_over(step1, STAGE_TEXT, out_dir=INTER_DIR, tag='', label='FULL stage 2')
_chk = level2.merge(step1[[ID_COL, 'level1']], on=ID_COL)
_invalid = _chk[[b not in L1_TO_L2.get(a, []) for a, b in zip(_chk['level1'], _chk['level2'])]]
print(f'Level 2 assigned: {len(level2):,} of {len(step1):,}; invalid/blank: {len(_invalid)}')
assert (_invalid['level2'] == '').all(), 'a non-blank Level 2 outside its Level 1 list'
assert len(_invalid) / max(len(level2), 1) <= 0.005, 'Level 2 blank rate above 0.5%'

## 11. Merge everything back onto all tickets

In [ ]:
EMPTY_L1, EMPTY_L2 = 'Unclear / No Query', 'Empty Chat / No Content'
assert EMPTY_L2 in L1_TO_L2[EMPTY_L1]

merged = step1.merge(level2, on=ID_COL, how='left')
merged['classification_source'] = 'llm'
merged.loc[merged['level1'].isin(DETERMINISTIC_L2), 'classification_source'] = 'llm:l1 + rule:l2'

rule_frame = pd.DataFrame({
    ID_COL: rule_assigned[ID_COL].values,
    'level1_code': LABEL_TO_L1_CODE[EMPTY_L1], 'level1': EMPTY_L1,
    'level1_justification': (f'No subject and under {SHORT_CHAR_THRESHOLD} characters of '
                             f'conversation: assigned by rule without an LLM call.'),
    'level1_evidence': '', 'level1_evidence_grounded': pd.NA,
    'level2_code': '', 'level2': EMPTY_L2,
    'level2_justification': 'Assigned by the same rule as Level 1.',
    'level2_evidence': '', 'level2_evidence_grounded': pd.NA, 'level2_module': '',
    'level2_override': '', 'classification_source': 'rule:empty_ticket',
})
labels_all = pd.concat([merged, rule_frame], ignore_index=True)
for c in ('level2_code', 'level2', 'level2_justification', 'level2_evidence', 'level2_module',
          'level2_override'):
    labels_all[c] = labels_all[c].fillna('')

META_COLS = [ID_COL, 'Incident_ID', 'Channel', DATE_COL, 'month', SUBJECT_COL, 'agent_group_name',
             'customer_classification_type', ORACLE_L1, ORACLE_L2, 'resolution_code_lvl1_desc',
             'conversation_chars_original', 'customer_chars', 'excel_cell_capped', TEXT_COL]
final = tickets[META_COLS].merge(labels_all, on=ID_COL, how='left')
final['was_truncated_for_model'] = final[ID_COL].isin(
    set(to_classify.loc[to_classify['was_truncated'], ID_COL]))
final['access_unknown_rootcause'] = final['level2'] == UNKNOWN_ACCESS_L2

assert len(final) == len(tickets) and final[ID_COL].is_unique
assert final['level1'].notna().all() and (final['level1'] != '').mean() >= 0.995
_ok = [b in L1_TO_L2.get(a, []) for a, b in zip(final['level1'], final['level2'])]
print(f'Final: {len(final):,} rows; L1/L2 valid pairs: {sum(_ok):,} '
      f'({sum(_ok) / len(final) * 100:.2f}%)')
print(final['classification_source'].value_counts().to_string())

## 12. Write the deliverable

`runs/output/scopus_2026_l1_l2_classification.xlsx` plus a companion CSV with full text.

In [ ]:
XL_LIMIT = 32_767
OUT_COLS = META_COLS[:-1] + [
    'level1_code', 'level1', 'level1_justification', 'level1_evidence', 'level1_evidence_grounded',
    'level2_code', 'level2', 'level2_justification', 'level2_evidence', 'level2_evidence_grounded',
    'access_unknown_rootcause', 'level2_override', 'level2_module', 'classification_source',
    'was_truncated_for_model', TEXT_COL]
deliverable = final[OUT_COLS].copy()
deliverable.to_csv(FINAL_CSV, index=False, encoding='utf-8-sig')


def share_table(s: pd.Series, name: str, denom: int) -> pd.DataFrame:
    vc = s.replace('', '(blank)').value_counts()
    return pd.DataFrame({name: vc.index, 'tickets': vc.values,
                         'share_%': (vc.values / max(denom, 1) * 100).round(1)})


N = len(deliverable)
l1_sum = share_table(deliverable['level1'], 'Level 1', N)
l1_sum['order'] = l1_sum['Level 1'].map({l: i for i, l in enumerate(L1_ORDER, 1)})

l2_rows = []
for l1 in L1_ORDER:
    sub = deliverable[deliverable['level1'] == l1]
    if sub.empty:
        continue
    t = share_table(sub['level2'], 'Level 2', len(sub)).rename(columns={'share_%': 'share_within_L1_%'})
    t.insert(0, 'Level 1', l1)
    t['share_of_total_%'] = (t['tickets'] / N * 100).round(2)
    l2_rows.append(t)
l2_sum = pd.concat(l2_rows, ignore_index=True)

_acc = deliverable[deliverable['level1'].isin(['Access', 'Scopus AI Access'])]
access_l2 = pd.crosstab(_acc['level2'], _acc['level1'], margins=True, margins_name='Total')
access_l2 = access_l2.reindex([*ACCESS_L2, UNKNOWN_ACCESS_L2, 'Total']).fillna(0).astype(int)
access_l2['note'] = ['' if i in ACCESS_L2 else
                     ('outside the 11 - no cause stated' if i == UNKNOWN_ACCESS_L2 else '')
                     for i in access_l2.index]

l1_vs_oracle = pd.crosstab(deliverable['level1'], deliverable[ORACLE_L1])
l1_by_month = pd.crosstab(deliverable['level1'], deliverable['month'])
l1_by_channel = pd.crosstab(deliverable['level1'], deliverable['Channel'])

tax_rows = [{'module': m, 'group': node.get('label', ''), 'code': c, 'label': leaf.get('label', ''),
             'guidance': str(leaf.get('description', ''))[:XL_LIMIT]}
            for m in ALL_MODULES
            for node in json.loads((TAXONOMY_DIR / f'{m}.json').read_text(encoding='utf-8')).values()
            for c, leaf in node.get('children', {}).items()]

_src = deliverable['classification_source'].value_counts()
notes = pd.DataFrame([
    ('Population', f'{N:,} Scopus tickets closed in {TARGET_YEAR}, from {TICKETS_XLSX.name}.'),
    ('Rules', f'Level 1 definitions and decision rules verbatim from {RULES_XLSX.name}; Level 2 '
              f'values from its Root Cause Descriptions sheet.'),
    ('Revised taxonomy', 'Level 1 is the revised 16-category list. Account Management folded into '
                         'Access (account changes) and Data / Privacy Request (personal data).'),
    ('Revised overrides (authored)', REVISED_OVERRIDES),
    ('Access addendum (authored)', ACCESS_ADDENDUM),
    *[(f'Access L2 addendum: {k} (authored)', v) for k, v in ACCESS_L2_ADDENDA.items()],
    ('Data / Privacy Request (authored)', DATA_PRIVACY_ADDENDUM),
    ('Access L2 precedence (authored)', ACCESS_L2_PRECEDENCE),
    *[(f'Unclear L2 addendum: {k} (authored)', v) for k, v in UNCLEAR_L2_ADDENDA.items()],
    ('Resolved/Thank-you correction (rule)', f'{int((deliverable["level2_override"] != "").sum())} '
     'tickets the model put in Resolved / Thank-you Only had no customer text at all, so they '
     'were set to Empty Chat / No Content (that value requires the customer to confirm or thank). '
     'Flagged in level2_override.'),
    *[('Taxonomy note', n) for n in TAXONOMY_NOTES],
    ('Shared rule row', 'The workbook defines Content / Indexing and Citation / Metrics in one row; '
                        'each category was given the half of that row tagged with its name.'),
    ('Unknown rootcause', 'Access / Scopus AI Access tickets with no cause stated anywhere get '
                          '"Unknown rootcause", outside the 11 values; flagged in '
                          'access_unknown_rootcause and shown separately in the Access L2 sheet.'),
    ('Two-stage design', 'Stage 1 assigns Level 1; tickets are routed by Level 1 code to a module '
                         "holding only that category's Level 2 values, so invalid pairs cannot occur."),
    ('Model input', f'"Subject: <incident_subject>" then the conversation. Conversations over '
                    f'{TRUNC_CAP:,} chars cut to the first {TRUNC_HEAD:,} + last {TRUNC_TAIL:,} '
                    f'({int(deliverable["was_truncated_for_model"].sum())} tickets). '
                    f'{int(deliverable["excel_cell_capped"].sum())} conversations were already cut '
                    f'at the Excel cell limit in the source file.'),
    ('Rule-assigned', f'{int(_src.get("rule:empty_ticket", 0))} tickets with no subject and under '
                      f'{SHORT_CHAR_THRESHOLD} characters of conversation -> {EMPTY_L1} / {EMPTY_L2}.'),
    ('Validation', 'No gold labels exist. Gates: smoke test (no invalid codes), 300-ticket sample '
                   f'for SME review ({SAMPLE_XLSX.name}). Accuracy has not been measured.'),
    ('Blank Level 2', f'{int((deliverable["level2"] == "").sum())} tickets have a Level 1 but no '
                      f'Level 2: the model returned no code (content-free ticket in a group with '
                      f'no Unknown value). Left blank rather than guessed.'),
    ('Evidence grounding', '*_evidence_grounded False = quote not found verbatim in the transcript; '
                           'NA = no quote offered.'),
    ('Oracle crosstab', 'L1 vs Oracle compares against the original Oracle contact reason tags, '
                        'which use a different taxonomy. It is a sanity check, not an accuracy score.'),
], columns=['topic', 'detail'])

manifest = pd.DataFrame(
    [('source_file', TICKETS_XLSX.name), ('rules_file', RULES_XLSX.name), ('rows', f'{N:,}'),
     *[(f'rows_{k}', f'{v:,}') for k, v in _src.items()],
     ('definition_workbook', DEFN_XLSX.name),
     ('generated_at', pd.Timestamp.now().isoformat(timespec='seconds'))]
    + [(f'fingerprint_{m}', PROMPT_FINGERPRINTS[m]) for m in ALL_MODULES],
    columns=['key', 'value'])


def fit_excel(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df[TEXT_COL] = df[TEXT_COL].astype(str).str.slice(0, XL_LIMIT - 1)
    return df


with pd.ExcelWriter(FINAL_XLSX, engine='openpyxl') as w:
    fit_excel(deliverable).to_excel(w, sheet_name='Tickets', index=False)
    l1_sum.sort_values('order').drop(columns='order').to_excel(w, sheet_name='L1 summary', index=False)
    l2_sum.to_excel(w, sheet_name='L2 by L1', index=False)
    access_l2.to_excel(w, sheet_name='Access L2')
    l1_vs_oracle.to_excel(w, sheet_name='L1 vs Oracle')
    l1_by_month.to_excel(w, sheet_name='L1 by month')
    l1_by_channel.to_excel(w, sheet_name='L1 by channel')
    pd.DataFrame(tax_rows).to_excel(w, sheet_name='Taxonomy used', index=False)
    notes.to_excel(w, sheet_name='Notes', index=False)
    manifest.to_excel(w, sheet_name='Run manifest', index=False)

print(f'Workbook : {FINAL_XLSX}\nFull text: {FINAL_CSV}\n')
print(l1_sum.sort_values('order').drop(columns='order').to_string(index=False))
print()
print(access_l2.to_string())

## Afterwards — restoring the previous config

Section 3 replaced `generated/pipeline_config.json` with the 15 Scopus modules. The previous
config and manifest are in `generated/_backup_pre_scopus2026/` — see `RESTORE.txt` there.